## Hospital Patient Data Analysis
## 1.Load the patient dataset and show summary with info().

In [126]:
import pandas as pd
import numpy as np

In [127]:
df_p = pd.read_csv("Patient_Data.csv")
df_b = pd.read_csv("Billing_Data.csv")

In [128]:
df_p.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PatientID       6 non-null      int64  
 1   Name            6 non-null      object 
 2   Department      6 non-null      object 
 3   Doctor          6 non-null      object 
 4   BillAmount      4 non-null      float64
 5   ReceptionistID  6 non-null      int64  
 6   CheckInTime     6 non-null      object 
dtypes: float64(1), int64(2), object(4)
memory usage: 464.0+ bytes


## 2.Select only the columns relevant for billing: ['PatientID', 'Department', 'Doctor', 'BillAmount'].

In [129]:
billing_cols = ['PatientID', 'Department', 'Doctor', 'BillAmount']
pb_df = df_p[billing_cols]
pb_df.head()


,PatientID,Department,Doctor,BillAmount
0,101,Cardiology,Dr. Smith,5000.0
1,102,Neurology,Dr. John,NaN
2,103,Orthopedics,Dr. Lee,7500.0
3,104,Cardiology,Dr. Smith,6200.0
4,105,Dermatology,Dr. Rose,NaN


## 3.Drop administrative columns like ['ReceptionistID', 'CheckInTime'].

In [111]:
df_p

,PatientID,Department,Doctor,BillAmount
0,101,Cardiology,Dr. Smith,5000.000000
1,102,Neurology,Dr. John,6233.333333
2,103,Orthopedics,Dr. Lee,7500.000000
3,104,Cardiology,Dr. Smith,6200.000000
4,105,Dermatology,Dr. Rose,6233.333333


In [130]:
df_p = df_p.drop(columns=['ReceptionistID', 'CheckInTime'])
df_p.head()

,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0
1,102,Bob,Neurology,Dr. John,NaN
2,103,Charlie,Orthopedics,Dr. Lee,7500.0
3,104,David,Cardiology,Dr. Smith,6200.0
4,105,Eva,Dermatology,Dr. Rose,NaN


## 4.Use groupby to find total bill amount per department.

In [131]:
dept_bill =pb_df.groupby('Department')['BillAmount'].sum()
dept_bill
## grouped patient by department
## Calculate department-wise money income.

Department
Cardiology     16200.0
Dermatology        0.0
Neurology          0.0
Orthopedics     7500.0
Name: BillAmount, dtype: float64

## 5.Remove duplicate patient records based on PatientID.

In [132]:
df_p = df_p.drop_duplicates(subset='PatientID')
df_p
## 101	Alice is removed.

,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0
1,102,Bob,Neurology,Dr. John,NaN
2,103,Charlie,Orthopedics,Dr. Lee,7500.0
3,104,David,Cardiology,Dr. Smith,6200.0
4,105,Eva,Dermatology,Dr. Rose,NaN


## 6.Fill missing BillAmount values with the mean bill amount.

In [133]:
## using mean filling all NaN values
mean_bill =df_p['BillAmount'].mean()
df_p['BillAmount'].fillna(mean_bill,inplace=True)
## calculates the average bill amount
## Replaces missing billing values using fillna()
df_p

/var/folders/67/sx853xbd5sn6l3qm1k3t6y9r0000gn/T/ipykernel_47425/1715320246.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_p['BillAmount'].fillna(mean_bill,inplace=True)


,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.000000
1,102,Bob,Neurology,Dr. John,6233.333333
2,103,Charlie,Orthopedics,Dr. Lee,7500.000000
3,104,David,Cardiology,Dr. Smith,6200.000000
4,105,Eva,Dermatology,Dr. Rose,6233.333333


## 7.Merge the billing dataset with patient dataset on PatientID.

In [134]:
merged_df =pd.merge(df_p,df_b, on='PatientID' , how='inner')
merged_df.head()
## inner join keeps valid matched records.

,PatientID,Name,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount
0,101,Alice,Cardiology,Dr. Smith,5000.000000,2000,3000
1,102,Bob,Neurology,Dr. John,6233.333333,1500,3500
2,103,Charlie,Orthopedics,Dr. Lee,7500.000000,2500,5000
3,104,David,Cardiology,Dr. Smith,6200.000000,3000,3200
4,105,Eva,Dermatology,Dr. Rose,6233.333333,1000,4000


## 8.Concatenate an additional DataFrame that contains new patients for the current week (row-wise).

In [135]:
# now new patients has to be created using dictionary[key:value] paris.
new_p = pd.DataFrame({
    'PatientID':[201,202],
    'Department':['Neurology','Cardiology'],
    'Doctor':['dr.sony','dr.singh'],
    'BillAmount':[12000,10000]
})

In [ ]:
merged_df=pd.concat(
[merged_df,new_p],
axis=0,
ignore_index=True)
## concat ---> combine DataFrames 
## axis=0 ---> row-wise concatenation
## ignore_index=True ---> resets the index after merging, new index 0 to n-1.

## 9.Concatenate new billing category columns like ['InsuranceCovered', 'FinalAmount'] (column-wise).

In [138]:
bill_categories =pd.DataFrame({ ## new df created from dictionary[key:value]pairs
'InsuranceCovered' : ['Yes'] * len(merged_df),  ## insurance is applied to all patients, if applied 'yes'.
    'FinalAmount': merged_df['BillAmount'] * 0.9 ## takes the billamount column from 'merged_df' and multiplies each values by 0.9
}) ## means discount 10%

final_dataset = pd.concat( ## combines df columns-wise
    [merged_df, bill_categories],
    axis=1 ## axis=1 adds columns side by side.
)

final_dataset.head() ## displays the first 5 rows of the final df, 'head' used for quickly verify.

,PatientID,Name,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount,InsuranceCovered,FinalAmount
0,101,Alice,Cardiology,Dr. Smith,5000.000000,2000.0,3000.0,Yes,4500.0
1,102,Bob,Neurology,Dr. John,6233.333333,1500.0,3500.0,Yes,5610.0
2,103,Charlie,Orthopedics,Dr. Lee,7500.000000,2500.0,5000.0,Yes,6750.0
3,104,David,Cardiology,Dr. Smith,6200.000000,3000.0,3200.0,Yes,5580.0
4,105,Eva,Dermatology,Dr. Rose,6233.333333,1000.0,4000.0,Yes,5610.0


## The hospital billing dataset was cleaned, merged using PatientID, and enhanced with accurate billing and insurance details. 

## The final dataset is reliable and ready for further analysis such as department-wise revenue and doctor performance evaluation.